# EDA Lab + Homework — AI Lab Instructor Notebook

*Python & Data · Easy*

Build strong intuition for Exploratory Data Analysis (EDA) as it’s used in real ML work: validating data assumptions, finding leakage, discovering slices, and making high-signal plots and summaries.

**Outcome.** Students can quickly diagnose dataset issues (missingness, outliers, imbalance, shift), produce defensible summary tables/plots, and articulate next modeling steps.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
$f

## Task 1: Dataset & Setup

# EDA Lab + Homework (Student)

**Goal**: practice high-signal EDA like you would in a FAANG ML interview or on-call investigation.

Rules:
- Work top-to-bottom
- Don't hardcode outputs
- Prefer concise, high-signal plots

## 1) Sanity Checks — 10 minutes

In [ ]:
$11

In [ ]:
# Checks
check('n_rows', df.shape[0] == 20)
check('has_target', 'churned' in df.columns)

**Why this works.** # EDA Lab + Homework (Instructor)

Solutions + short rationales. Keep answers concise and interview-oriented.

## Solutions

### 1.1 Grain + schema

Grain: **one row per user** (user-level snapshot).

### 1.2 Missingness + duplicates

### 2.1 Numeric summaries

### 2.2 High-signal plots

**One-line summary**

- Skew -> log transform
- Heavy tail -> clip + log
- Large range -> scale

### 3.1 Label imbalance

### 3.2 Slice analysis

### 4.1 Leakage-prone features

## Example Homework Write-up (brief)

- Missingness: `age` and `discount_pct` have NaNs; decide imputation policy.
- Potential data quality: some users have 0 sessions but high tenure; verify instrumentation.
- Imbalance: churn rate should be tracked; avoid accuracy as sole metric.
- Slices: churn differs by plan/country/tenure bucket; consider stratified evaluation.
- Leakage: `refund_after_churn_flag` must be excluded from training features.

## Task 2: Data grain + schema

## 1) Sanity Checks — 10 minutes

### Task 1.1: Data grain + schema

**Hints**
- churned should be int/bool; categorical columns should remain object/category

In [ ]:
# Grain: one row per user (user-level snapshot). Each row aggregates recent behavior,
# account state, and the churn label for a single user.

df['churned'] = df['churned'].astype(int)
df['last_payment_failed'] = df['last_payment_failed'].astype(int)
df['refund_after_churn_flag'] = df['refund_after_churn_flag'].astype(int)

df.dtypes
# Note: missing numeric values become NaN; that's expected for EDA.


## Task 3: Missingness + duplicates

### Task 1.2: Missingness + duplicates

In [ ]:
def missingness_table(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    n_missing = df.isna().sum()
    pct = n_missing / n
    out = pd.DataFrame({"n_missing": n_missing, "pct_missing": (n_missing / n)})
    return out.sort_values("pct_missing", ascending=False)

miss = missingness_table(df)
print(miss)
check('miss_has_cols', set(miss.columns) >= {'n_missing','pct_missing'})

n_dup_rows = int(df.duplicated().sum())
n_dup_user_id = int(df["user_id"].duplicated().sum())
print(n_dup_rows, n_dup_user_id)


In [ ]:
# Checks
check('miss_has_cols', set(miss.columns) >= {'n_missing','pct_missing'})

## Task 4: Numeric summaries

## 2) Distributions + Outliers — 15 minutes

### Task 2.1: Numeric summaries

**Hints**
- Select only Numeric Columns

In [ ]:
num = df.select_dtypes(include=[np.number])
summary = num.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
summary


## Task 5: Plot 2 high-signal distributions

### Task 2.2: Plot 2 high-signal distributions

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(df["sessions_last_7d"], bins=10)
plt.title("sessions_last_7d (skew)")
plt.subplot(1,2,2)
plt.hist(df["avg_session_min"], bins=10)
plt.title("avg_session_min (heavy tail)")
plt.tight_layout()
plt.show()


## Task 6: Label imbalance

## 3) Target + Slices — 15 minutes

### Task 3.1: Label imbalance

In [ ]:
churn_rate = float(df["churned"].mean())
counts = df["churned"].value_counts()
print("churn_rate:", churn_rate)
counts
# Rationale: if churn is rare, accuracy is misleading; prefer PR-AUC / F1 / cost-sensitive metrics.


**Why this works.** if churn is rare, accuracy is misleading; prefer PR-AUC / F1 / cost-sensitive metrics.

For imbalance, inspect churn rate and class counts first; accuracy alone can be misleading when the positive class is rare.

## Task 7: Slice analysis

### Task 3.2: Slice analysis

**Hints**
- pd.cut

In [ ]:
by_country = df.groupby("country")["churned"].mean().sort_values(ascending=False)
by_plan = df.groupby("plan")["churned"].mean().sort_values(ascending=False)
by_country, by_plan

df2 = df.copy()
df2["tenure_bucket"] = pd.cut(df2["tenure_days"], bins=[-1, 14, 60, 180, 10_000], labels=["<=2w","2w-2m","2m-6m",">6m"])
by_bucket = df2.groupby("tenure_bucket")["churned"].mean().sort_values(ascending=False)
by_bucket


## Task 8: Identify leakage-prone features

## 4) Leakage + Time — 10 minutes

### Task 4.1: Identify leakage-prone features

**Hints**
- refund_after_churn_flag looks suspicious.

In [ ]:
leaky_cols = ["refund_after_churn_flag"]
leaky_cols
# Rationale: this column is defined after churn outcome; it would not exist at prediction time.
# It can trivially predict churn (near-perfect correlation), inflating offline metrics.


**Why this works.** this column is defined after churn outcome; it would not exist at prediction time.